In [ ]:
# Se importan las librerías necesarias para el proyecto
import requests                 #Para hacer peticiones HTTP ("abrir" las páginas web)
from bs4 import BeautifulSoup   #Para analizar o "leer" el contenido HTML de las páginas web
import pandas as pd             #Para manejar los datos en forma de tablas (DataFrames) 
import time                     #Para pausar el programa entre peticiones y no sobrecargar el servidor  

In [ ]:
# Se define la URL base del sitio web y los encabezados para simular un navegador
BASE_URL = "https://www.beer-analytics.com"
HEADERS  = {"User-Agent": "Mozilla/5.0 (portfolio-project/1.0)"}

In [ ]:
# Se define una función para obtener los estilos de cerveza desde la página principal
def obtener_estilos(url_principal):
    resp = requests.get(url_principal, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(resp.text, "html.parser")

    estilos = {}
    for link in soup.find_all("a", href=True):
        href  = link["href"]
        texto = link.get_text(strip=True)

        # Filtramos solo estilos individuales (3 niveles) y descartamos categorías (2 niveles)
        # /styles/ipa/                → 3 partes → categoría, no nos sirve
        # /styles/ipa/american-ipa/  → 4 partes → estilo individual, nos sirve
        partes = href.strip("/").split("/")
        if partes[0] == "styles" and len(partes) == 3 and texto:
            url_completa = BASE_URL + href
            if url_completa not in estilos.values():   # evitar duplicados
                estilos[texto] = url_completa

    print(f"Estilos encontrados: {len(estilos)}")
    return estilos




In [ ]:
# Se define una función para extraer las especificaciones de cada estilo de cerveza
def extraer_specs(nombre, url):
    resp = requests.get(url, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(resp.text, "html.parser")

    datos = {"Estilo": nombre, "URL": url}

    # Extraer Origin, Fermentation, Color, Strength, Taste, Recipes, etc.
    for dt in soup.find_all("dt"):
        campo = dt.get_text(strip=True).replace(":", "")
        dd    = dt.find_next("dd")
        if dd:
            datos[campo] = dd.get_text(strip=True)

    # Extraer especificaciones numéricas (ABV, IBU, Color EBC, OG)
    for h3 in soup.find_all("h3"):
        titulo  = h3.get_text(strip=True)
        parrafo = h3.find_next("p")
        if not parrafo:
            continue
        texto        = parrafo.get_text(" ", strip=True)
        texto_limpio = texto.replace("Specification", "").strip()
        partes       = texto_limpio.split("–")

        if len(partes) >= 2:
            try:
                minimo   = float(partes[0].strip().split()[0])
                maximo   = float(partes[1].strip().split()[0])
                promedio = round((minimo + maximo) / 2, 2)
            except ValueError:
                continue

            if titulo == "Alcohol":
                datos["ABV_min_%"]  = minimo
                datos["ABV_max_%"]  = maximo
                datos["ABV_prom_%"] = promedio
            elif titulo == "Bitterness":
                datos["IBU_min"]  = minimo
                datos["IBU_max"]  = maximo
                datos["IBU_prom"] = promedio
            elif titulo == "Color":
                datos["Color_EBC_min"] = minimo
                datos["Color_EBC_max"] = maximo
            elif titulo == "Original Extract":
                datos["OG_min"] = minimo
                datos["OG_max"] = maximo

    return datos

In [ ]:
# Se obtiene el diccionario de estilos y se itera para extraer las especificaciones de cada uno
estilos = obtener_estilos(BASE_URL + "/styles/")

registros = []
total = len(estilos)


for i, (nombre, url) in enumerate(estilos.items(), start=1):
    print(f"[{i}/{total}] Scraping: {nombre}...")
    try:
        reg = extraer_specs(nombre, url)
        registros.append(reg)
    except KeyboardInterrupt:
        # Si presionas stop, guarda lo que lleva antes de salir
        print("\n⚠️ Interrumpido! Guardando lo que se logró scraping...")
        break
    except Exception as e:
        print(f"  ERROR en {nombre}: {e}")

    time.sleep(0.5)  # ← espera 0.5 segundos entre cada petición

# Guardar aunque esté incompleto
df = pd.DataFrame(registros)
df.to_csv("beer_styles_todos.csv", index=False)
print(f"\nArchivo guardado con {len(df)} estilos!")

In [ ]:
# ── PASO 4: Guardar todos los estilos ─────────────────────────────────────
print(f"\nEstilos totales scrapeados: {len(df)}")

cols_vista = ["Estilo", "Origin", "Fermentation", "Color", "Strength", "Taste",
              "ABV_min_%", "ABV_max_%", "ABV_prom_%",
              "IBU_min",   "IBU_max",   "IBU_prom"]

print("\n", df[cols_vista].to_string(index=False))

df.to_csv("beer_styles_todos.csv", index=False)
print("\nArchivo guardado!")

